[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/03-data-engineering/de-databases.ipynb)

# Databases — SQL vs NoSQL

*AIBits Academy · Machine Learning End To End · Data Engineering*

Where data lives before and after you touch it. Choosing the right store shapes everything downstream.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

## Two Families of Databases

Most acquired data ends up in a **database** — a system built for storing, querying, and updating data reliably at scale. Databases split into two broad families, and picking the right one depends on the shape of your data and the queries you'll run.

|  | SQL (Relational) | NoSQL (Non-relational) |
|---|---|---|
| **Structure** | Fixed schema — every row has the same columns | Schema-less — documents can differ from one another |
| **Best for** | Structured data, complex queries, transactions | Unstructured / semi-structured data, huge volume, horizontal scale |
| **Scaling** | Typically vertical (a bigger server) | Horizontal (spread across many servers) |
| **Guarantees** | **ACID** transactions | Flexible consistency (often eventual) |
| **Examples** | MySQL, PostgreSQL, SQLite, Oracle, SQL Server | MongoDB, Cassandra, Redis, CouchDB, Neo4j |

> **🔒 ACID — Why Relational Databases Are Trusted for Money**
>
> A **transaction** is a group of operations treated as one all-or-nothing unit. ACID is the four guarantees that make them safe: **A**tomicity (all steps succeed or none do — a bank transfer never debits without crediting), **C**onsistency (the database moves from one valid state to another), **I**solation (concurrent transactions don't corrupt each other), **D**urability (once committed, it survives a crash). This is exactly why banks and order systems live on SQL.

## SQL in Python — the built-in `sqlite3`

**SQLite** is a full SQL database that lives in a single file with no separate server — and Python ships with the `sqlite3` module built in, so it's the perfect place to learn. The workflow is always: **connect → cursor → execute → commit → close**. Here we set up two tables for a Surat textile firm, Mehta Textiles:

Start from a clean file so re-running the notebook is always safe.

In [ ]:
import os
if os.path.exists('mehta.db'):
    os.remove('mehta.db')

In [ ]:
import sqlite3

conn = sqlite3.connect('mehta.db')   # connect (creates the file if absent)
c = conn.cursor()                       # cursor executes SQL

c.execute("""CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY, name TEXT, salary REAL, dept_id INTEGER)""")
c.execute("""CREATE TABLE departments (
    dept_id INTEGER PRIMARY KEY, dept_name TEXT)""")

c.executemany("INSERT INTO departments VALUES (?,?)",
              [(1,'Weaving'), (2,'Dyeing'), (3,'Sales')])
c.executemany("INSERT INTO employees (name, salary, dept_id) VALUES (?,?,?)",
              [('Anjali',72000,1), ('Rahul',58000,2),
               ('Priya',91000,3), ('Vikram',45000,1), ('Neha',67000,3)])

conn.commit()   # save changes — nothing persists until you commit

> **⚠ Always Use Parameterised Queries (?)**
>
> Notice the values go in as `(?, ?)` placeholders, never glued into the SQL string with f-strings. Building SQL by string-concatenating user input is the classic **SQL-injection** vulnerability. The `?` form lets the database driver escape values safely — make it a reflex.

## Querying — SELECT, WHERE, ORDER BY, JOIN, GROUP BY

A SQL query reads almost like an English sentence. This one says "give me the name and salary of employees earning over 60,000, highest first":

In [ ]:
rows = c.execute("""SELECT name, salary FROM employees
                    WHERE salary > 60000
                    ORDER BY salary DESC""").fetchall()
for r in rows:
    print(r)

The real power of relational databases is the **JOIN** — combining rows from two tables on a shared column (here `dept_id`). This attaches each employee's department name from the `departments` table:

In [ ]:
rows = c.execute("""SELECT e.name, d.dept_name
                    FROM employees e
                    INNER JOIN departments d ON e.dept_id = d.dept_id
                    ORDER BY e.name""").fetchall()
for r in rows:
    print(r)

There are four join flavours, differing only in which unmatched rows they keep:

And **GROUP BY** collapses rows into summary groups — the SQL equivalent of pandas' `groupby`. Average salary per department:

In [ ]:
rows = c.execute("""SELECT d.dept_name, ROUND(AVG(e.salary), 0) AS avg_sal
                    FROM employees e
                    JOIN departments d ON e.dept_id = d.dept_id
                    GROUP BY d.dept_name
                    ORDER BY avg_sal DESC""").fetchall()
for r in rows:
    print(r)
conn.close()

## NoSQL — When Rows & Columns Don't Fit

Not all data is tabular. A product catalogue where every item has different attributes, a stream of JSON events, a social graph of who-follows-whom — these strain the fixed-schema relational model. NoSQL databases relax it, and come in four main types:

| Type | Stores data as | Examples | Ideal for |
|---|---|---|---|
| **Document** | JSON-like documents | MongoDB, CouchDB | Content management, catalogues, flexible records |
| **Key-Value** | Simple key → value pairs | Redis, DynamoDB | Caching, sessions, ultra-fast lookups |
| **Wide-Column** | Columns grouped together | Cassandra, HBase | Analytics over huge datasets |
| **Graph** | Nodes & edges | Neo4j, ArangoDB | Deeply interconnected data (social, fraud rings) |

The most common is the **document** store, MongoDB. Instead of rows, you insert Python dictionaries — and different documents in the same collection can have different fields, which is impossible in a rigid SQL table:

> **Run this one on your own computer (terminal or local Jupyter), not in Colab** (it is a shell command, or needs a desktop window, a running server, or keyboard input).

```python
# pip install pymongo — connects to a running MongoDB server
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['mehta']
products = db['products']

# Two documents with DIFFERENT fields — perfectly legal in a document store
products.insert_many([
    {'name': 'Cotton Saree', 'price': 1200, 'colours': ['red', 'blue']},
    {'name': 'Silk Dupatta', 'price': 850, 'in_stock': True, 'gsm': 90},
])

# Query with a dict filter — find products under ₹1000
for p in products.find({'price': {'$lt': 1000}}):
    print(p['name'], p['price'])
```

> **🧭 SQL or NoSQL — A Quick Rule of Thumb**
>
> Reach for **SQL** when your data is naturally tabular, relationships matter, and correctness of transactions is critical (finance, orders, inventory). Reach for **NoSQL** when records are irregular or schema-fluid, you need to scale writes across many machines, or you're caching for speed. Many real systems use *both* — SQL for the money, NoSQL for the catalogue and cache.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Filter and sort with SQL

Using the in-memory table `orders`, write a query that returns the `customer` and `amount` of orders above 500, highest first, into `rows` (a list of tuples).

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
c = conn.cursor()
c.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer TEXT, amount REAL)")
c.executemany("INSERT INTO orders (customer, amount) VALUES (?, ?)", [("Asha", 900), ("Ben", 300), ("Asha", 650), ("Chen", 520), ("Ben", 80)])
rows = None   # TODO: rows = c.execute("...").fetchall()


In [ ]:
try:
    check("three big orders, highest first", rows == [("Asha", 900.0), ("Asha", 650.0), ("Chen", 520.0)])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import sqlite3
conn = sqlite3.connect(":memory:")
c = conn.cursor()
c.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, customer TEXT, amount REAL)")
c.executemany("INSERT INTO orders (customer, amount) VALUES (?, ?)", [("Asha", 900), ("Ben", 300), ("Asha", 650), ("Chen", 520), ("Ben", 80)])
rows = c.execute("SELECT customer, amount FROM orders WHERE amount > 500 ORDER BY amount DESC").fetchall()

```

</details>

### Exercise 2 · Medium · Join and aggregate

Add a `customers` table (`name`, `city`) and write one query that returns each **city** with the **total** order amount, largest total first, into `by_city`.

In [ ]:
c.execute("CREATE TABLE customers (name TEXT, city TEXT)")
c.executemany("INSERT INTO customers VALUES (?, ?)", [("Asha", "Pune"), ("Ben", "Surat"), ("Chen", "Pune")])
by_city = None   # TODO (join orders to customers, GROUP BY city)


In [ ]:
try:
    check("city totals", by_city == [("Pune", 2070.0), ("Surat", 380.0)])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
c.execute("CREATE TABLE customers (name TEXT, city TEXT)")
c.executemany("INSERT INTO customers VALUES (?, ?)", [("Asha", "Pune"), ("Ben", "Surat"), ("Chen", "Pune")])
by_city = c.execute("""SELECT cu.city, SUM(o.amount) AS total FROM orders o
                     JOIN customers cu ON cu.name = o.customer
                     GROUP BY cu.city ORDER BY total DESC""").fetchall()

```

</details>

### Exercise 3 · Stretch · Parameterised queries (and why)

Write `orders_over(conn, amount)` using a `?` placeholder (never string formatting). It must return matching rows and must be safe against input such as `"0 OR 1=1"`.

In [ ]:
def orders_over(conn, amount):
    pass   # TODO


In [ ]:
try:
    check("normal use", len(orders_over(conn, 500)) == 3)
    check("injection attempt matches nothing", len(orders_over(conn, "0 OR 1=1")) == 0)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def orders_over(conn, amount):
    return conn.execute("SELECT customer, amount FROM orders WHERE amount > ?", (amount,)).fetchall()

```

With a placeholder the driver treats the whole input as a value, so it can never change the shape of the query.

</details>

---
*Back to the course: **Machine Learning End To End → Databases — SQL vs NoSQL**.*